In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import numpy as np

import pandas as pd

import yfinance as yf

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report

In [4]:
ticker = "AAPL"

data = yf.download(ticker, start="2021-01-01", end="2026-01-01")

/tmp/ipykernel_49/4228461109.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start="2021-01-01", end="2026-01-01")
[*********************100%***********************]  1 of 1 completed


In [5]:
print(data)

Price            Close        High         Low        Open     Volume
Ticker            AAPL        AAPL        AAPL        AAPL       AAPL
Date                                                                 
2021-01-04  125.632523  129.709922  123.059875  129.622552  143301900
2021-01-05  127.185776  127.894478  124.681085  125.127664   97664900
2021-01-06  122.904495  127.224602  122.690916  123.991805  155088000
2021-01-07  127.098427  127.787708  124.127751  124.613156  109578200
2021-01-08  128.195435  128.758506  126.428554  128.564332  105158200
...                ...         ...         ...         ...        ...
2025-12-24  273.066711  274.682309  271.461097  271.600701   17910600
2025-12-26  272.657867  274.622521  272.119325  273.415814   21521800
2025-12-29  273.016876  273.615223  271.610700  271.949773   23715200
2025-12-30  272.338684  273.335969  271.540868  272.069428   22139600
2025-12-31  271.122009  272.937076  271.012322  272.318764   27293600

[1255 rows x 5 colu

In [6]:
if isinstance(data.columns, pd.MultiIndex):

    data.columns = data.columns.get_level_values(0)



In [7]:
df = data[['Open', 'High', 'Low', 'Close', 'Volume']].copy()

In [8]:
df.head(20)

Price,Open,High,Low,Close,Volume
Date,,,,,
2021-01-04,129.622552,129.709922,123.059875,125.632523,143301900
2021-01-05,125.127664,127.894478,124.681085,127.185776,97664900
2021-01-06,123.991805,127.224602,122.690916,122.904495,155088000
2021-01-07,124.613156,127.787708,124.127751,127.098427,109578200
2021-01-08,128.564332,128.758506,126.428554,128.195435,105158200
2021-01-11,125.418919,126.370308,124.749058,125.215042,100384500
2021-01-12,124.749068,125.904334,123.156940,125.040314,91951100
2021-01-13,125.001453,127.612934,124.739345,127.069283,88636800
2021-01-14,126.981956,127.176115,125.001495,125.147125,90221800


In [9]:
df['Return'] = np.log(df['Close'] / df['Close'].shift(1))



# Create predictive features (predictors)

df['SMA_5'] = df['Close'].rolling(5).mean() / df['Close']

df['SMA_20'] = df['Close'].rolling(20).mean() / df['Close']

df['Vol_RSI'] = df['Volume'].rolling(5).mean() / df['Volume'].rolling(20).mean()



In [10]:
df.head()

Price,Open,High,Low,Close,Volume,Return,SMA_5,SMA_20,Vol_RSI
Date,,,,,,,,,
2021-01-04,129.622552,129.709922,123.059875,125.632523,143301900,NaN,NaN,NaN,NaN
2021-01-05,125.127664,127.894478,124.681085,127.185776,97664900,0.012288,NaN,NaN,NaN
2021-01-06,123.991805,127.224602,122.690916,122.904495,155088000,-0.034241,NaN,NaN,NaN
2021-01-07,124.613156,127.787708,124.127751,127.098427,109578200,0.033554,NaN,NaN,NaN
2021-01-08,128.564332,128.758506,126.428554,128.195435,105158200,0.008594,0.98446,NaN,NaN


In [11]:
df['Target'] = np.where(df['Return'].shift(-1) > 0, 1, 0)

df.dropna(inplace=True)



In [12]:
features = ['SMA_5', 'SMA_20', 'Vol_RSI', 'Return']

X = df[features]

y = df['Target']



In [13]:
split_idx = int(len(df) * 0.4)
k = int(len(df) * 0.6)



X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:k]

y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:k]



In [14]:
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

model.fit(X_train, y_train)

RandomForestClassifier(max_depth=5, random_state=42)

In [15]:
test_preds = model.predict(X_test)

print(f"--- Model Performance Matrix ({ticker}) ---")

print(f"Out-of-Sample Accuracy: {accuracy_score(y_test, test_preds):.2%}")

--- Model Performance Matrix (AAPL) ---
Out-of-Sample Accuracy: 47.77%


In [16]:
from sklearn.svm import SVC

In [17]:
svm_model = SVC(kernel='rbf', C=1.0, gamma='scale')

# Train the model on the scaled training data
svm_model.fit(X_train, y_train)

SVC()

In [18]:
predictions = svm_model.predict(X_test)

# Evaluate model performance
print("Accuracy:", accuracy_score(y_test, predictions))

Accuracy: 0.5506072874493927


In [19]:
backtest_df = df.iloc[split_idx:k].copy()
backtest_df.head()

Price,Open,High,Low,Close,Volume,Return,SMA_5,SMA_20,Vol_RSI,Target
Date,,,,,,,,,,
2023-01-18,134.420695,136.179299,132.662077,132.838928,69672800,-0.005384,0.995207,0.968708,0.868793,1
2023-01-19,131.728739,133.860683,131.424177,132.897873,58280400,0.000444,0.997398,0.969350,0.851410,1
2023-01-20,132.907723,135.599680,131.866314,135.452301,80223600,0.019039,0.985058,0.953090,0.873266,1
2023-01-23,135.697880,140.806704,135.481737,138.635452,81760300,0.023228,0.971441,0.933212,0.939309,1
2023-01-24,137.849508,140.649536,137.839688,140.030579,66435100,0.010013,0.971010,0.927527,0.953950,0


In [20]:
backtest_df['Signal'] = predictions

In [21]:
backtest_df['Strategy_Return'] = backtest_df['Signal'].shift(1) * backtest_df['Return']

In [22]:
backtest_df.head(10)

Price,Open,High,Low,Close,Volume,Return,SMA_5,SMA_20,Vol_RSI,Target,Signal,Strategy_Return
Date,,,,,,,,,,,,
2023-01-18,134.420695,136.179299,132.662077,132.838928,69672800,-0.005384,0.995207,0.968708,0.868793,1,1,NaN
2023-01-19,131.728739,133.860683,131.424177,132.897873,58280400,0.000444,0.997398,0.969350,0.851410,1,1,0.000444
2023-01-20,132.907723,135.599680,131.866314,135.452301,80223600,0.019039,0.985058,0.953090,0.873266,1,1,0.019039
2023-01-23,135.697880,140.806704,135.481737,138.635452,81760300,0.023228,0.971441,0.933212,0.939309,1,1,0.023228
2023-01-24,137.849508,140.649536,137.839688,140.030579,66435100,0.010013,0.971010,0.927527,0.953950,0,1,0.010013
2023-01-25,138.419293,139.932280,136.375767,139.372284,65799300,-0.004712,0.984971,0.935433,0.942330,1,1,-0.004712
2023-01-26,140.659316,141.720378,139.411583,141.435471,54105100,0.014695,0.982676,0.926625,0.940536,1,1,0.014695
2023-01-27,140.649512,144.648132,140.570913,143.370926,70555800,0.013592,0.980456,0.920931,0.923711,0,1,0.013592
2023-01-30,142.417945,142.997595,140.344946,140.492310,64015300,-0.020282,1.003189,0.944483,0.882342,1,1,-0.020282


In [23]:
tc = 0.001 

backtest_df['Trades'] = backtest_df['Signal'].diff().fillna(0).abs()

backtest_df['Strategy_Return_Net'] = backtest_df['Strategy_Return'] - (backtest_df['Trades'] * tc)

In [24]:
backtest_df.head(10)

Price,Open,High,Low,Close,Volume,Return,SMA_5,SMA_20,Vol_RSI,Target,Signal,Strategy_Return,Trades,Strategy_Return_Net
Date,,,,,,,,,,,,,,
2023-01-18,134.420695,136.179299,132.662077,132.838928,69672800,-0.005384,0.995207,0.968708,0.868793,1,1,NaN,0.0,NaN
2023-01-19,131.728739,133.860683,131.424177,132.897873,58280400,0.000444,0.997398,0.969350,0.851410,1,1,0.000444,0.0,0.000444
2023-01-20,132.907723,135.599680,131.866314,135.452301,80223600,0.019039,0.985058,0.953090,0.873266,1,1,0.019039,0.0,0.019039
2023-01-23,135.697880,140.806704,135.481737,138.635452,81760300,0.023228,0.971441,0.933212,0.939309,1,1,0.023228,0.0,0.023228
2023-01-24,137.849508,140.649536,137.839688,140.030579,66435100,0.010013,0.971010,0.927527,0.953950,0,1,0.010013,0.0,0.010013
2023-01-25,138.419293,139.932280,136.375767,139.372284,65799300,-0.004712,0.984971,0.935433,0.942330,1,1,-0.004712,0.0,-0.004712
2023-01-26,140.659316,141.720378,139.411583,141.435471,54105100,0.014695,0.982676,0.926625,0.940536,1,1,0.014695,0.0,0.014695
2023-01-27,140.649512,144.648132,140.570913,143.370926,70555800,0.013592,0.980456,0.920931,0.923711,0,1,0.013592,0.0,0.013592
2023-01-30,142.417945,142.997595,140.344946,140.492310,64015300,-0.020282,1.003189,0.944483,0.882342,1,1,-0.020282,0.0,-0.020282


In [25]:
backtest_df['Market_CuReturn'] = np.exp(backtest_df['Return'].cumsum())

backtest_df['Strategy_CuReturn'] = np.exp(backtest_df['Strategy_Return_Net'].cumsum())

In [26]:
backtest_df.head(15)

Price,Open,High,Low,Close,Volume,Return,SMA_5,SMA_20,Vol_RSI,Target,Signal,Strategy_Return,Trades,Strategy_Return_Net,Market_CuReturn,Strategy_CuReturn
Date,,,,,,,,,,,,,,,,
2023-01-18,134.420695,136.179299,132.662077,132.838928,69672800,-0.005384,0.995207,0.968708,0.868793,1,1,NaN,0.0,NaN,0.994630,NaN
2023-01-19,131.728739,133.860683,131.424177,132.897873,58280400,0.000444,0.997398,0.969350,0.851410,1,1,0.000444,0.0,0.000444,0.995072,1.000444
2023-01-20,132.907723,135.599680,131.866314,135.452301,80223600,0.019039,0.985058,0.953090,0.873266,1,1,0.019039,0.0,0.019039,1.014198,1.019673
2023-01-23,135.697880,140.806704,135.481737,138.635452,81760300,0.023228,0.971441,0.933212,0.939309,1,1,0.023228,0.0,0.023228,1.038032,1.043636
2023-01-24,137.849508,140.649536,137.839688,140.030579,66435100,0.010013,0.971010,0.927527,0.953950,0,1,0.010013,0.0,0.010013,1.048478,1.054138
2023-01-25,138.419293,139.932280,136.375767,139.372284,65799300,-0.004712,0.984971,0.935433,0.942330,1,1,-0.004712,0.0,-0.004712,1.043549,1.049183
2023-01-26,140.659316,141.720378,139.411583,141.435471,54105100,0.014695,0.982676,0.926625,0.940536,1,1,0.014695,0.0,0.014695,1.058997,1.064714
2023-01-27,140.649512,144.648132,140.570913,143.370926,70555800,0.013592,0.980456,0.920931,0.923711,0,1,0.013592,0.0,0.013592,1.073488,1.079284
2023-01-30,142.417945,142.997595,140.344946,140.492310,64015300,-0.020282,1.003189,0.944483,0.882342,1,1,-0.020282,0.0,-0.020282,1.051935,1.057614


In [27]:
final_market = (backtest_df['Market_CuReturn'].iloc[-1] - 1) * 100

final_strategy = (backtest_df['Strategy_CuReturn'].iloc[-1] - 1) * 100



print("\n--- Backtest Strategy Metrics ---")

print(f"Buy & Hold Absolute Return: {final_market:.2f}%")

print(f"ML Strategy Net Absolute Return: {final_strategy:.2f}%")


--- Backtest Strategy Metrics ---
Buy & Hold Absolute Return: 37.73%
ML Strategy Net Absolute Return: 41.53%


In [28]:
strat_std = backtest_df['Strategy_Return_Net'].std()

sharpe = (backtest_df['Strategy_Return_Net'].mean() / strat_std) * np.sqrt(252) if strat_std != 0 else 0

print(f"Annualized Sharpe Ratio: {sharpe:.2f}")

Annualized Sharpe Ratio: 1.82
